# Outline
The purpose of this script is to conduct time-domain analyses on mode timecourses
and generate figures

In [ ]:
# Imports
import os
import mne
import re
import pandas as pd
import numpy as np
import glob
import datetime
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import yaml
from nilearn import image, datasets
from plotting_functions import *  # custom functions for plotting modes
import seaborn as sns
from scipy.stats import gaussian_kde, zscore, ttest_rel
from sklearn.preprocessing import StandardScaler

import arviz as az

# Imports for R functionality
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

base = importr('base')
BF = importr('BayesFactor')

# Suppress mne output
mne.set_log_level('WARNING')

In [ ]:
# Write out data?
writeOK=False

In [ ]:
# Get current date for output files
now = datetime.datetime.now()
datetime_str = now.strftime("%Y-%m-%d_%H%M")

In [ ]:
# Define functions

# Define a helper function to pull the most recent version of a file
# Note that this sleects file based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

# Define a function for shading timepoints within a boolean mask. Used later for plotting. 
def shade_timepoints(mask):
    shaded_timepoints = []
    is_shaded = False
    start = None
    for i, val in enumerate(mask):
        if val and not is_shaded:
            start = i
            is_shaded = True
        elif not val and is_shaded:
            shaded_timepoints.append((start, i - 1))
            is_shaded = False
    if is_shaded:
        shaded_timepoints.append((start, len(mask) - 1))
    return shaded_timepoints

In [ ]:
# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']
#model_dir = os.path.join('/d', 'mjt', '9', 'projects', 'Dynamo-Analysis', 'POND-EGNG', 'results_N=92', '6_modes')

# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
exclude = config['misc']['exclusion_col']
groups = config['misc']['groups']

# Update model path according to run #
dynemo_run = 'run3'
model_dir = f'{model_dir}_{dynemo_run}'

# Define directory containing posthoc output
posthoc_dir = os.path.join(model_dir, 'posthoc')

# Other params
orig_mode_order = config['posthoc']['orig_mode_order']
new_mode_order = config['posthoc']['new_mode_order']
mode_order = [orig_mode_order.index(name) for name in new_mode_order]
colours_new_order = [colours[i] for i in mode_order]

In [ ]:
# Files to load
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt')) 
CNg_timecourses_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_timecourses.npy')) 
happy_timecourses_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_happy_timecourses.npy'))
angry_timecourses_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_angry_timecourses.npy'))

times_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_times.npy'))
ages_fname = recent_fname(os.path.join(posthoc_dir, '*_all_subject_ages.npy'))

# Load list of subjects that went into DyNeMo
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

# Read the vector of ages
ages = np.load(ages_fname).squeeze()

# Read the CNg timecourses and times for each mode
mean_timecourses = np.load(CNg_timecourses_fname)  # shape (subjects, modes, timepoints)
times = np.load(times_fname)

# Also read happy/angry timecourses
mean_happy_timecourses = np.load(happy_timecourses_fname)
mean_angry_timecourses = np.load(angry_timecourses_fname)

# CNg timecourse analysis
Compare each post-stimulus timepoint tothe pre-stimulus period

In [ ]:
baseline = (times>-0.2) * (times < -0.1)

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(12,6), gridspec_kw={'height_ratios': [2, 1]})
all_bfs = [] # stores all bfs across modes

for mode_idx in range(len(mode_order)):

    # Plot mode averages in the top axis, BF's in the lower axis
    ax_av = axs[0]
    ax_bf = axs[1]

    # Compute and plot mode average (corrected for baseline)
    mode_timecourse = mean_timecourses[:,mode_idx,:]
    values = np.mean(mode_timecourse, 0)
    values -= np.mean(values[baseline])
    err = np.std(mode_timecourse, 0)/ np.sqrt(mode_timecourse.shape[0])
    line, = ax_av.plot(times, values, label=orig_mode_order[mode_idx], color=colours[mode_idx])
    colours.append(line.get_color())  # store line colours for later plots
    ax_av.fill_between(times, values-err, values+err, alpha=0.2, color=colours[mode_idx])

    # Loop through time points, performing BF tests against baseline
    bfs = []
    base = ro.FloatVector(mode_timecourse[:, baseline].mean(axis=1))  # baseline is the mean over the baseline time points

    for t in range(mode_timecourse.shape[1]):
        x = ro.FloatVector(mode_timecourse[:, t])

        bf_test = BF.ttestBF(x, base, paired=True)
        bf_value = BF.extractBF(bf_test).rx2("bf")[0]
        bfs.append(bf_value)

    # Plot BF timecourse in the lower axis
    bfs = np.array(bfs)
    all_bfs.append(bfs)
    ax_bf.plot(times, bfs, color=colours[mode_idx])

    # Loop through timepoints again, and mark those where BF exceeds the 95th percentile of BFS for this mode     
    thresh = np.percentile(bfs, 90)
    for t, bf in enumerate(bfs):
        if bf >= thresh:
            
            # Vertical offset based on mode index so they stack vertically
            v_pos = 0.95 - (mode_idx * 0.05)

            ax_av.text(times[t], v_pos, '-', fontsize=20, color=colours[mode_idx],
                       transform=ax_av.get_xaxis_transform()) # forces scale-agnostic position

    # # Alternatively, pu a marker at every timepoint, but make the opacity proportional to BF
    # log_bfs = np.log10(bfs)

    # for t, bf in enumerate(log_bfs):
    #     if bf > 1:  # only plot markers for BF > 1

    #         # Compute % of the max bf at this timepoint, and use this to set the opacity of the marker
    #         opacity = bf / np.max(log_bfs)
    #         v_pos = 0.95 - (mode_idx * 0.02)
    #         ax_av.text(times[t], v_pos+0.04, '-', fontsize=20, color=colours[mode_idx], alpha=opacity,
    #                    transform=ax_av.get_xaxis_transform()) # forces scale-agnostic position
            
# Top axis aesthetics
ax_av.set_xlabel('Time (s)', fontsize=12)
ax_av.set_xticks([-0.2, 0, 0.25, 0.5, 0.75, 1.0])
ax_av.set_ylabel('Mode Activation', fontsize=12)
ax_av.spines['top'].set_visible(False)
ax_av.spines['right'].set_visible(False)
ax_av.spines['bottom'].set_visible(False)
ax_av.spines['left'].set_visible(False)
ax_av.axvline(0, linestyle='--', color=[0.2,0.2,0.2, 0.5], label=None) # line at t=0

# Lower axis aesthetics
ax_bf.axhline(1, color='k', linestyle='--', linewidth=1) # line at BF=1.0
ax_bf.set_xlabel('Time (s)', fontsize=12)
ax_bf.set_ylabel('log(BF$_{10}$)', fontsize=12)
ax_bf.spines['top'].set_visible(False)
ax_bf.spines['right'].set_visible(False)
ax_bf.spines['bottom'].set_visible(False)
ax_bf.spines['left'].set_visible(False)

# Arrange legend in logical order
handles, labels = ax_av.get_legend_handles_labels()
ax_bf.set_yscale('log')

# Legend
order = [labels.index(name) for name in new_mode_order]
handles_ord = [handles[i] for i in order]
labels_ord  = [labels[i] for i in order]

fig.legend(
    handles_ord,
    labels_ord,
    ncol=1,
    fontsize=10,
    frameon=False,
    loc='center',
    bbox_to_anchor=(0.8, 0.1, 0.8, 0.1)
    )

plt.tight_layout()
plt.show() 

# Happy/angry timecourse analysis
Compare mode activation for happy vs angry faces, at each timepoint

In [ ]:
baseline = (times>-0.2) * (times < -0.1)

fig, axs = plt.subplots(
    nrows=2,
    ncols=2,  # use 6 for all modes
    figsize=(12, 6),
    sharex=False,
    sharey=False,
    gridspec_kw={'height_ratios': [2, 1]}

)

# Select modes to analyze and plot
modes = [mode_order[0], mode_order[4]] # 0 and 4 correspond to modes 3 and 5 (visual & frontotemporal)

# Loop through modes
for panel_idx, mode_idx in enumerate(modes):   # replace "modes" with "mode order" to plot all modes  

    # Select the appropriate timecourses
    mode_happy_timecourse = mean_happy_timecourses[:,mode_idx]
    mode_angry_timecourse = mean_angry_timecourses[:,mode_idx]

    # Apply baseline correction (for plotting only!) and compute SE's
    h_values = np.mean(mode_happy_timecourse, 0)
    h_values -= np.mean(h_values[baseline])

    a_values = np.mean(mode_angry_timecourse, 0)
    a_values -= np.mean(a_values[baseline])

    h_err = np.std(mode_happy_timecourse, 0)/ np.sqrt(mode_happy_timecourse.shape[0])
    a_err = np.std(mode_angry_timecourse, 0)/ np.sqrt(mode_angry_timecourse.shape[0])

    
    # Define an empty lists of BFs _for this mode_
    bfs = []

    # Loop through time points
    for t in range(mode_happy_timecourse.shape[1]):

        # Define happy and angry amplitudes at this time point
        h = ro.FloatVector(mode_happy_timecourse[:, t])
        a = ro.FloatVector(mode_angry_timecourse[:, t])

        # Perform a paired Bayes t-test between happy and angry
        bf_test = BF.ttestBF(h, a, paired=True)
        bf_value = BF.extractBF(bf_test).rx2("bf")[0]

        # Store the results
        bfs.append(bf_value)

    # Top row: Mean (over subjects) happy and angry timecourses
    ax_tc = axs[0, panel_idx]
    ax_tc.set_title(f'{orig_mode_order[mode_idx]} Mode', fontsize=14)
    ax_tc.set_xlabel('Time(s)')

    # Happy
    ax_tc.plot(times, h_values, color=colours[mode_idx], linestyle='-', label="H")
    ax_tc.fill_between(times, h_values-h_err, h_values+h_err, color=colours[mode_idx], alpha=0.25)
    
    # Angry
    ax_tc.plot(times, a_values, color=colours[mode_idx], linestyle='--', label="A")
    ax_tc.fill_between(times, a_values-a_err, a_values+a_err, color=colours[mode_idx], alpha=0.25)

    # Show legend
    # ax_tc.legend(ncol=2, columnspacing=1.5, bbox_to_anchor=(0.6, 1.0), loc='upper left', frameon=False, fontsize=12) # horixzontal legend
    # ax_tc.legend(ncol=1, labelspacing=2, bbox_to_anchor=(0.8, 0.95), loc='upper left', frameon=False, fontsize=12) # vertical legend

    # # Modify linewidth in legend
    # leg = ax_tc.get_legend()
    # for line in leg.get_lines():
    #     line.set_linewidth(2.0)
    
    # Bottom row: Bayes Factors
    ax_bf = axs[1, panel_idx]
    ax_bf.sharey(axs[1, 0])   # force shared y axis with the first column
    
    
    ax_bf.plot(times, bfs, color='k')  #colours[mode_idx])
    ax_bf.axhline(1, color='k', linestyle='-', linewidth=1)
    ax_bf.set_ylim(None, 2000)
    ax_bf.set_yscale('log')
    ax_bf.set_xlabel('Time(s)')
    
    
    # Set y-axis labels only for the first plot
    if panel_idx==0:
        ax_tc.set_ylabel('Mode Amplitude')
        ax_bf.set_ylabel('log (BF$_{10}$)')
        
    else:
        ax_tc.set_ylabel('')
        ax_bf.set_ylabel('')

    # Add vertical lines to denote t=0
    ax_tc.axvline(0, linestyle='--', color='k', linewidth=1)
    ax_bf.axvline(0, linestyle='--', color='k', linewidth=1)

plt.tight_layout()
plt.show()


# Happy/angry timecourse analysis with NHST
For supplementary materials

In [ ]:
fig, axs = plt.subplots(
    nrows=2,
    ncols=3,  
    figsize=(12, 6),
)

axs = axs.flatten()

# Loop through modes
for panel_idx, mode_idx in enumerate(mode_order):

    # Select the appropriate timecourses
    mode_happy_timecourse = mean_happy_timecourses[:,mode_idx]
    mode_angry_timecourse = mean_angry_timecourses[:,mode_idx]

    # Apply baseline correction (for plotting only!) and compute SE's
    h_values = np.mean(mode_happy_timecourse, 0)
    h_values -= np.mean(h_values[baseline])

    a_values = np.mean(mode_angry_timecourse, 0)
    a_values -= np.mean(a_values[baseline])

    h_err = np.std(mode_happy_timecourse, 0)/ np.sqrt(mode_happy_timecourse.shape[0])
    a_err = np.std(mode_angry_timecourse, 0)/ np.sqrt(mode_angry_timecourse.shape[0])

    
    # Define an empty lists of p values _for this mode_
    p_values = []

    # Loop through time points
    for t in range(mode_happy_timecourse.shape[1]):
        
        # Define happy and angry amplitudes at this time point
        h = mode_happy_timecourse[:, t]
        a = mode_angry_timecourse[:, t]

        # Perform a t-test between happy and angry and store the p value
        p_value = ttest_rel(h, a).pvalue
        p_values.append(p_value)

    # Apply FDR correction
    p_arr = np.asarray(p_values)
    _, p_arr_fdr = mne.stats.fdr_correction(p_arr, alpha=0.01)

    # Get masks for shading
    sig_mask_uncorrected = p_arr < 0.01
    sig_mask_fdr = p_arr_fdr < 0.01
    
    if np.any(sig_mask_fdr):
        shaded_timepoints = shade_timepoints(sig_mask_fdr)
    else:
        shaded_timepoints = shade_timepoints(sig_mask_uncorrected)

    # The frontotemporal mode doesn't survive FDR. We'll show uncorrected shading in the figure
    if mode_idx==0:
        sig_mask_uncorrected = p_arr < 0.01
        shaded_timepoints = shade_timepoints(sig_mask_uncorrected)

    # Plot mean (over subjects) happy and angry timecourses, and shade times where p_fdr < 0.05
    ax = axs[panel_idx]

    ax.set_title(f'{orig_mode_order[mode_idx]} Mode', fontsize=14)
    ax.set_xlabel('Time(s)')

    # Happy
    ax.plot(times, h_values, color=colours[mode_idx], linestyle='-', label="H")
    ax.fill_between(times, h_values-h_err, h_values+h_err, color=colours[mode_idx], alpha=0.25)
    
    # Angry
    ax.plot(times, a_values, color=colours[mode_idx], linestyle='--', label="A")
    ax.fill_between(times, a_values-a_err, a_values+a_err, color=colours[mode_idx], alpha=0.25)
    
    # Shade significant time points 
    for start_idx, end_idx in shaded_timepoints:
        ax.axvspan(
            times[start_idx],
            times[end_idx],
            alpha=0.25,
            linewidth=0,
            color=colours[mode_idx]
        )

    # If there are significant time points, add a star to the plot
    if np.any(sig_mask_fdr):
        ax.text(0.7, 0.9, r'*', transform=ax.transAxes, fontsize=15, ha='left', va='center')
        #ax.text(0.7, 0.9, r'$p_{FDR} < 0.01$', transform=ax.transAxes, fontsize=10, ha='left', va='center')  # option to have explicit label
    elif np.any(sig_mask_uncorrected):
        ax.text(0.7, 0.9, r'$\dagger$', transform=ax.transAxes, fontsize=15, ha='left', va='center')

plt.tight_layout()
plt.show()

# CNg ~ Age analysis
Correlate mode activation with age at every timepoint. For supplementary materials. 

In [ ]:
thresh = 90  # top 10% for shading

fig, axs = plt.subplots(
    nrows=2,
    ncols=6,
    figsize=(18, 6),
    #sharex=True,
    sharey='row'
)

for panel_idx, mode_idx in enumerate(mode_order):  # plot panels in the desired order

    bfs = []
    correlations = []

    for t in range(mean_timecourses.shape[2]):
        x = ro.FloatVector(mean_timecourses[:, mode_idx, t])
        y = ro.FloatVector(ages)

        # Bayesian correlation
        bf_test = BF.correlationBF(x, y)
        bf_value = BF.extractBF(bf_test).rx2("bf")[0]
        bfs.append(bf_value)

        # Pearson correlation
        r = np.corrcoef(mean_timecourses[:, mode_idx, t], ages)[0, 1]
        correlations.append(r)

    bfs = np.array(bfs)
    correlations = np.array(correlations)

    # --- BF mask for shading: top 10% AND BF >= 10 ---
    bf_mask = (bfs >= np.percentile(bfs, thresh)) & (bfs >= 10.0)
    shaded_timepoints = shade_timepoints(bf_mask)


    # Top row: Pearson correlations
    ax_r = axs[0, panel_idx]
    ax_r.plot(times, correlations, color=colours[mode_idx])
    ax_r.axhline(0, color='k', linestyle='--', linewidth=1)
    ax_r.set_title(f'{orig_mode_order[mode_idx]} Mode', fontsize=14)
    ax_r.set_xlabel('Time', fontsize=14)

    # Bottom row: Bayes Factors
    ax_bf = axs[1, panel_idx]
    ax_bf.plot(times, bfs, color=colours[mode_idx])
    ax_bf.axhline(1, color='k', linestyle='--', linewidth=1)
    ax_bf.set_yscale('log')
    ax_bf.set_xlabel('Time', fontsize=14)

    # Shade correlation plot where BF is strong
    for start_idx, end_idx in shaded_timepoints:
        ax_r.axvspan(
            times[start_idx],
            times[end_idx],
            alpha=0.25,
            linewidth=0,
            color=colours[mode_idx]
        )

# Axis labels
axs[0, 0].set_ylabel('Pearson r', fontsize=14)
axs[1, 0].set_ylabel('log(BF$_{10}$)', fontsize=14)

plt.tight_layout()
plt.show()

